In [1]:
import pandas as pd

In [11]:
import kagglehub

path = kagglehub.dataset_download("vivek468/superstore-dataset-final")
print(path)

/kaggle/input/datasets/vivek468/superstore-dataset-final


In [12]:
import os

for root, dirs, files in os.walk(path):
    print(files)

['Sample - Superstore.csv']


In [14]:
print(path)

/kaggle/input/datasets/vivek468/superstore-dataset-final


In [16]:
import pandas as pd

df = pd.read_csv(
    "/kaggle/input/datasets/vivek468/superstore-dataset-final/Sample - Superstore.csv",
    encoding="latin1"
)

In [17]:
print(df.columns)

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='object')


In [21]:
import duckdb

In [22]:
con = duckdb.connect(database=':memory:')
con.register('superstore_table', df)

In [33]:
# Apply WHERE Filters
#Filtering data to find high-value orders ($\ge 500$) within the 'Technology' category in the 'West' region.

query_step_3 ="""
SELECT 
    "Order ID", 
    "Order Date", 
    "Customer Name", 
    "Region", 
    "Category", 
    "Sales"
FROM superstore_table
WHERE "Region" = 'West'
  AND "Category" = 'Technology'
  AND "Sales" >= 500;
"""
print("--- Filtered Sub-segment (High-value West Tech Orders) ---")

# Execute the query string and convert the result directly to a DataFrame
display(con.execute(query_step_3).df())

--- Filtered Sub-segment (High-value West Tech Orders) ---


,Order ID,Order Date,Customer Name,Region,Category,Sales
0,CA-2014-115812,6/9/2014,Brosina Hoffman,West,Technology,907.152
1,CA-2014-115812,6/9/2014,Brosina Hoffman,West,Technology,911.424
2,CA-2016-145625,9/11/2016,Kelly Collister,West,Technology,3347.370
3,CA-2015-137946,9/1/2015,Doug Bickford,West,Technology,959.984
4,US-2014-119137,7/23/2014,Arthur Gainer,West,Technology,1023.936
...,...,...,...,...,...,...
121,CA-2014-145254,7/23/2014,Nick Crebassa,West,Technology,604.752
122,CA-2015-156566,10/1/2015,Eric Murdock,West,Technology,572.800
123,CA-2016-160717,6/6/2016,Maria Etezadi,West,Technology,3023.928
124,CA-2015-130113,12/27/2015,Aaron Hawkins,West,Technology,668.160


In [34]:
#Use GROUP BY for Aggregations
#Grouping performance by region and general product category to evaluate baseline business health.
query_step_4 = """
SELECT 
    "Region",
    "Category",
    SUM("Sales") AS "Total Sales",
    SUM("Quantity") AS "Total Quantity Sold",
    AVG("Sales") AS "Average Order Value",
    SUM("Profit") AS "Net Profit",
    AVG("Discount") * 100 AS "Avg Discount %"
FROM superstore_table
GROUP BY "Region", "Category"
ORDER BY "Region" ASC, "Total Sales" DESC;
"""
print("--- Regional & Category Performance Aggregations ---")
display(con.execute(query_step_4).df())

--- Regional & Category Performance Aggregations ---


,Region,Category,Total Sales,Total Quantity Sold,Average Order Value,Net Profit,Avg Discount %
0,Central,Technology,170416.3120,1544.0,405.753124,33697.4320,13.309524
1,Central,Office Supplies,167026.4150,5409.0,117.458801,8879.9799,25.274262
2,Central,Furniture,163797.1638,1827.0,340.534644,-2871.0494,29.738046
3,East,Technology,264973.9810,1942.0,495.278469,47462.0351,14.336449
4,East,Furniture,208291.2040,2214.0,346.574383,3046.1658,15.407654
5,East,Office Supplies,205516.0550,6462.0,120.044425,41014.5791,14.293224
6,South,Technology,148771.9080,1118.0,507.753952,19991.8314,10.784983
7,South,Office Supplies,125651.3130,3800.0,126.282727,19986.3928,16.743719
8,South,Furniture,117298.6840,1291.0,353.309289,6771.2061,12.153614
9,West,Furniture,252612.7435,2696.0,357.302325,11504.9503,13.140028


In [25]:
# Use Case A: Top 5 Products by Revenue
query_step_5a = """
SELECT 
    "Product ID", 
    "Product Name", 
    SUM("Sales") AS "Total Revenue"
FROM superstore_table
GROUP BY "Product ID", "Product Name"
ORDER BY "Total Revenue" DESC
LIMIT 5;
"""
print("--- Top 5 Revenue-Generating Products ---")
display(con.execute(query_step_5a).df())

# Use Case B: Top 5 Most Profitable Sub-Categories
query_step_5b = """
SELECT 
    "Sub-Category", 
    SUM("Profit") AS "Total Sub-Category Profit"
FROM superstore_table
GROUP BY "Sub-Category"
ORDER BY "Total Sub-Category Profit" DESC
LIMIT 5;
"""
print("\n--- Top 5 Most Profitable Sub-Categories ---")
display(con.execute(query_step_5b).df())

--- Top 5 Revenue-Generating Products ---


,Product ID,Product Name,Total Revenue
0,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,61599.824
1,OFF-BI-10003527,Fellowes PB500 Electric Punch Plastic Comb Bin...,27453.384
2,TEC-MA-10002412,Cisco TelePresence System EX90 Videoconferenci...,22638.480
3,FUR-CH-10002024,HON 5400 Series Task Chairs for Big and Tall,21870.576
4,OFF-BI-10001359,GBC DocuBind TL300 Electric Binding System,19823.479



--- Top 5 Most Profitable Sub-Categories ---


,Sub-Category,Total Sub-Category Profit
0,Copiers,55617.8249
1,Phones,44515.7306
2,Accessories,41936.6357
3,Paper,34053.5693
4,Binders,30221.7633


In [27]:
# Use Case A: Monthly Sales & Profit Trends (Fixed with strptime for MM/DD/YYYY)
query_step_6a = """
SELECT 
    STRFTIME(STRPTIME("Order Date", '%m/%d/%Y'), '%Y-%m') AS "Sales Month",
    SUM("Sales") AS "Monthly Revenue",
    SUM("Profit") AS "Monthly Profit"
FROM superstore_table
GROUP BY 1
ORDER BY "Sales Month" ASC;
"""
print("--- Month-over-Month Business Performance Trends ---")
display(con.execute(query_step_6a).df())

# Use Case B: Top 5 Spenders ("Customer Name" tracking)
query_step_6b = """
SELECT 
    "Customer ID",
    "Customer Name",
    SUM("Sales") AS "Total Customer Spend",
    SUM("Profit") AS "Total Profit Generated"
FROM superstore_table
GROUP BY "Customer ID", "Customer Name"
ORDER BY "Total Customer Spend" DESC
LIMIT 5;
"""
print("\n--- Top 5 Customers by Cumulative Value ---")
display(con.execute(query_step_6b).df())

# Use Case C: Duplicate Row Check
query_step_6c = """
SELECT "Row ID", COUNT(*) AS "Record Count"
FROM superstore_table
GROUP BY "Row ID"
HAVING COUNT(*) > 1;
"""
print("\n--- Duplicate Check Results (Empty means perfect row uniqueness) ---")
display(con.execute(query_step_6c).df())

--- Month-over-Month Business Performance Trends ---


,Sales Month,Monthly Revenue,Monthly Profit
0,2014-01,14236.8950,2450.1907
1,2014-02,4519.8920,862.3084
2,2014-03,55691.0090,498.7299
3,2014-04,28295.3450,3488.8352
4,2014-05,23648.2870,2738.7096
5,2014-06,34595.1276,4976.5244
6,2014-07,33946.3930,-841.4826
7,2014-08,27909.4685,5318.1050
8,2014-09,81777.3508,8328.0994
9,2014-10,31453.3930,3448.2573



--- Top 5 Customers by Cumulative Value ---


,Customer ID,Customer Name,Total Customer Spend,Total Profit Generated
0,SM-20320,Sean Miller,25043.050,-1980.7393
1,TC-20980,Tamara Chand,19052.218,8981.3239
2,RB-19360,Raymond Buch,15117.339,6976.0959
3,TA-21385,Tom Ashbrook,14595.620,4703.7883
4,AB-10105,Adrian Barton,14473.571,5444.8055



--- Duplicate Check Results (Empty means perfect row uniqueness) ---


,Row ID,Record Count


In [28]:
#Validate Results & Data Quality
#Executing a final data validation sanity check to ensure no anomalies undermine down-stream operational visualizations.
query_step_7 = """
SELECT 
    COUNT(*) AS "Total Row Count",
    COUNT("Row ID") AS "Valid Rows",
    COUNT("Order ID") AS "Valid Orders",
    COUNT(*) - COUNT("Customer Name") AS "Missing Customer Names",
    SUM(CASE WHEN "Sales" < 0 THEN 1 ELSE 0 END) AS "Negative Sales Records",
    SUM(CASE WHEN "Quantity" <= 0 THEN 1 ELSE 0 END) AS "Invalid Quantity Records"
FROM superstore_table;
"""
print("--- Data Pipeline Health & Validation Metrics ---")
display(con.execute(query_step_7).df())

--- Data Pipeline Health & Validation Metrics ---


,Total Row Count,Valid Rows,Valid Orders,Missing Customer Names,Negative Sales Records,Invalid Quantity Records
0,9994,9994,9994,0,0.0,0.0
